# GeoVision-CLIP — Muestreo estratificado guiado (Situacion 2)

Dataset: `juanjoseorozcolopez/geovision-fuentes`

Genera tiles 64x64x13 etiquetados en 5 clases:
- 3 contaminacion guiadas por S5P > p90/p99
- vegetacion_densa: random + NDVI > 0.6
- suelo_urbano: radio 1 km de estaciones DAGMA + NDVI < 0.3

Variables compartidas entre celdas: `s2 no2 so2 o3 era5 modis`, `times`,
`perc`, `aceptados`, `df_ctx`, `tiles_arr`, `meta`.

In [ ]:
from __future__ import annotations

import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

SEED = 42
TILE_PX = 64
N_BANDAS_S2 = 13
HALF = TILE_PX // 2
SCL_THRESHOLD = 0.3
VENTANA_S2_DIAS = 5
NDVI_URBANO_MAX = 0.30
RADIO_DAGMA_M = 1000

BASE_PATH = Path("/kaggle/input/datasets/juanjoseorozcolopez/geovision-fuentes")
OUT_DIR = Path("/kaggle/working")
CLASES = ["contaminacion_alta_NO2", "contaminacion_alta_SO2",
          "ozono_anomalo", "vegetacion_densa", "suelo_urbano"]

rng = np.random.default_rng(SEED)

## 1. Abrir paneles y cargar DAGMA

In [ ]:
def _open(name):
    return xr.open_zarr(BASE_PATH / name / "panel.zarr", consolidated=True)

s2    = _open("copernicus_s2_sr_harmonized")
no2   = _open("copernicus_s5p_offl_l3_no2")
so2   = _open("copernicus_s5p_offl_l3_so2")
o3    = _open("copernicus_s5p_offl_l3_o3")
era5  = _open("ecmwf_era5_hourly")
modis = _open("modis_061_mcd19a2_granules")

dagma_df   = pd.read_parquet(BASE_PATH / "dagma" / "dagma_cvc_horario_raw.parquet")
estaciones = pd.read_csv(BASE_PATH / "dagma" / "estaciones_metadata.csv")

bands_s2 = s2["band"].values.tolist()
y_coords = s2["y"].values
x_coords = s2["x"].values

print(f"S2:    {dict(s2.sizes)}")
print(f"NO2:   {dict(no2.sizes)} | SO2: {dict(so2.sizes)} | O3: {dict(o3.sizes)}")
print(f"ERA5:  {dict(era5.sizes)} | MODIS: {dict(modis.sizes)}")
print(f"DAGMA: {len(dagma_df):,} filas, {len(estaciones)} estaciones")

## 2. Parsear timestamps (cache por panel)

In [ ]:
def parse_time(s):
    s = str(s)
    # MODIS DOY: A2021001
    if len(s) == 8 and s[0] in "ATMP" and s[1:].isdigit():
        return pd.Timestamp(year=int(s[1:5]), month=1, day=1) + pd.Timedelta(days=int(s[5:8]) - 1)
    if "_" in s:
        s = s.split("_")[0]
    if len(s) >= 9 and s[8:9] == "T":
        if len(s) == 11:
            return pd.to_datetime(s + "0000", format="%Y%m%dT%H%M%S")
        return pd.to_datetime(s[:15], format="%Y%m%dT%H%M%S")
    return pd.to_datetime(s)

def _parse_all(panel):
    return pd.DatetimeIndex([parse_time(t) for t in panel["time"].values])

t0 = time.time()
times = {
    "s2":    _parse_all(s2),
    "no2":   _parse_all(no2),
    "so2":   _parse_all(so2),
    "o3":    _parse_all(o3),
    "era5":  _parse_all(era5),
    "modis": _parse_all(modis),
}
print(f"Parseados en {time.time()-t0:.1f}s")
for k, v in times.items():
    print(f"  {k:<6}: {v[0]} a {v[-1]}  ({len(v):,})")

## 3. Percentiles globales de S5P sobre Cali

In [ ]:
def percentiles_s5p(panel, banda, n_sample=500, filtrar_fillvalue=True):
    n_t = panel.sizes["time"]
    idx = np.sort(rng.choice(n_t, size=min(n_sample, n_t), replace=False))
    data = panel["data"].sel(band=banda).isel(time=idx).values.ravel()
    data = data[np.isfinite(data)]
    if filtrar_fillvalue:
        data = data[np.abs(data) > 1e-12]
    return {
        "p10": float(np.percentile(data, 10)),
        "p50": float(np.percentile(data, 50)),
        "p90": float(np.percentile(data, 90)),
        "p99": float(np.percentile(data, 99)),
        "n":   int(data.size),
    }

perc = {
    "NO2": percentiles_s5p(no2, "tropospheric_NO2_column_number_density"),
    "SO2": percentiles_s5p(so2, "SO2_column_number_density"),
    "O3":  percentiles_s5p(o3,  "O3_column_number_density"),
}
for g, p in perc.items():
    print(f"{g}: p10={p['p10']:.2e} p50={p['p50']:.2e} "
          f"p90={p['p90']:.2e} p99={p['p99']:.2e}  (n={p['n']:,})")

## 4. Helpers: TileCand, extracción, NDVI/NDBI/SCL, texto

In [ ]:
@dataclass
class Tile:
    t_idx: int
    time_s2: str
    y_idx: int
    x_idx: int
    lat: float
    lon: float
    clase: str = None
    no2: float = np.nan
    so2: float = np.nan
    o3: float = np.nan
    ndvi: float = np.nan
    ndbi: float = np.nan
    scl_pct: float = np.nan

def extraer(t_idx, y_idx, x_idx):
    return s2["data"].isel(
        time=t_idx,
        y=slice(y_idx - HALF, y_idx + HALF),
        x=slice(x_idx - HALF, x_idx + HALF),
    ).values

def ndvi_of(tile):
    nir = tile[bands_s2.index("B8")].astype("float64")
    red = tile[bands_s2.index("B4")].astype("float64")
    d = np.where((nir + red) == 0, np.nan, nir + red)
    return float(np.nanmean((nir - red) / d))

def ndbi_of(tile):
    swir = tile[bands_s2.index("B11")].astype("float64")
    nir = tile[bands_s2.index("B8")].astype("float64")
    d = np.where((swir + nir) == 0, np.nan, swir + nir)
    return float(np.nanmean((swir - nir) / d))

def scl_of(tile):
    scl = tile[bands_s2.index("SCL")]
    return float(np.isin(scl, [4, 5, 6, 7]).mean())

def texto_de(clase, no2, so2, o3, ndvi):
    return {
        "contaminacion_alta_NO2": f"Zona urbana con NO2 alto ({no2:.2e} mol/m2), trafico vehicular intenso.",
        "contaminacion_alta_SO2": f"Pluma industrial con SO2 elevado ({so2:.2e} mol/m2), corredor Yumbo-Acopi.",
        "ozono_anomalo":          f"Anomalia de ozono ({o3:.2e} mol/m2), fotoquimica activa.",
        "vegetacion_densa":       f"Vegetacion densa, NDVI={ndvi:.2f}, cana de azucar o bosque.",
        "suelo_urbano":           f"Zona urbana construida, NDVI={ndvi:.2f}, alta densidad edificada.",
    }[clase]

## 5. Muestreo guiado por S5P (3 clases de contaminacion)

Estrategia: indexar pixeles S5P > umbral, buscar S2 cerca (+-5 dias), validar SCL, extraer tile.

In [ ]:
def indexar_calientes(panel, banda, umbral):
    arr = panel["data"].sel(band=banda).values
    t_idx, y_idx, x_idx = np.where(np.isfinite(arr) & (arr > umbral))
    return t_idx, y_idx, x_idx, arr

def s2_cercanas(t_dt):
    delta = np.abs((times["s2"] - t_dt).total_seconds().values)
    cands = np.where(delta < VENTANA_S2_DIAS * 86400)[0]
    return cands[np.argsort(delta[cands])] if len(cands) else np.array([], dtype=int)

def muestrear_guiada(clase, n_obj, panel, banda, umbral, key_time, attr):
    t_hot, y_hot, x_hot, arr = indexar_calientes(panel, banda, umbral)
    print(f"  [{clase}] {len(t_hot):,} pixeles calientes")
    perm = rng.permutation(len(t_hot))
    aceptados_cls = []
    intentos = rj_no_s2 = rj_scl = rj_borde = 0
    t0 = time.time()
    for k in perm:
        intentos += 1
        if intentos > n_obj * 60 or len(aceptados_cls) >= n_obj:
            break
        t_idx_p = int(t_hot[k])
        lat_p = float(panel["y"].values[int(y_hot[k])])
        lon_p = float(panel["x"].values[int(x_hot[k])])
        cands_s2 = s2_cercanas(times[key_time][t_idx_p])
        if len(cands_s2) == 0:
            rj_no_s2 += 1; continue
        lat = lat_p + float(rng.uniform(-0.0018, 0.0018))
        lon = lon_p + float(rng.uniform(-0.0018, 0.0018))
        y = int(np.argmin(np.abs(y_coords - lat)))
        x = int(np.argmin(np.abs(x_coords - lon)))
        if not (HALF <= y < s2.sizes["y"] - HALF and HALF <= x < s2.sizes["x"] - HALF):
            rj_borde += 1; continue
        encontrado = False
        for t_idx_s2 in cands_s2[:5]:
            t_idx_s2 = int(t_idx_s2)
            tile = extraer(t_idx_s2, y, x)
            if tile.shape != (N_BANDAS_S2, TILE_PX, TILE_PX):
                continue
            scl = scl_of(tile)
            if scl < SCL_THRESHOLD:
                continue
            t = Tile(t_idx=t_idx_s2, time_s2=str(s2["time"].values[t_idx_s2]),
                     y_idx=y, x_idx=x, lat=float(y_coords[y]), lon=float(x_coords[x]),
                     clase=clase, scl_pct=scl, ndvi=ndvi_of(tile), ndbi=ndbi_of(tile))
            setattr(t, attr, float(arr[t_idx_p, int(y_hot[k]), int(x_hot[k])]))
            aceptados_cls.append((t, tile))
            encontrado = True
            break
        if not encontrado:
            rj_scl += 1
    print(f"  [{clase}] {len(aceptados_cls)}/{n_obj} en {time.time()-t0:.1f}s "
          f"(intentos={intentos}, rj_no_s2={rj_no_s2}, rj_scl={rj_scl}, rj_borde={rj_borde})")
    return aceptados_cls

Tamano del muestreo. Para test: 5; para full: 1000.

In [ ]:
N_POR_CLASE = 5

aceptados = {}

In [ ]:
aceptados["contaminacion_alta_NO2"] = muestrear_guiada(
    "contaminacion_alta_NO2", N_POR_CLASE, no2,
    "tropospheric_NO2_column_number_density",
    perc["NO2"]["p90"], "no2", "no2")

In [ ]:
aceptados["contaminacion_alta_SO2"] = muestrear_guiada(
    "contaminacion_alta_SO2", N_POR_CLASE, so2,
    "SO2_column_number_density",
    perc["SO2"]["p90"], "so2", "so2")

In [ ]:
aceptados["ozono_anomalo"] = muestrear_guiada(
    "ozono_anomalo", N_POR_CLASE, o3,
    "O3_column_number_density",
    perc["O3"]["p99"], "o3", "o3")

## 6. Muestreo aleatorio: vegetacion_densa

In [ ]:
def muestrear_aleatoria(clase, n_obj, criterio):
    aceptados_cls = []
    intentos = rj_scl = rj_clase = 0
    t0 = time.time()
    while len(aceptados_cls) < n_obj and intentos < n_obj * 80:
        intentos += 1
        t = int(rng.integers(0, s2.sizes["time"]))
        y = int(rng.integers(HALF, s2.sizes["y"] - HALF))
        x = int(rng.integers(HALF, s2.sizes["x"] - HALF))
        tile = extraer(t, y, x)
        if tile.shape != (N_BANDAS_S2, TILE_PX, TILE_PX):
            continue
        scl = scl_of(tile)
        if scl < SCL_THRESHOLD:
            rj_scl += 1; continue
        ndvi = ndvi_of(tile)
        ndbi = ndbi_of(tile)
        if criterio(ndvi, ndbi):
            aceptados_cls.append((Tile(
                t_idx=t, time_s2=str(s2["time"].values[t]),
                y_idx=y, x_idx=x, lat=float(y_coords[y]), lon=float(x_coords[x]),
                clase=clase, scl_pct=scl, ndvi=ndvi, ndbi=ndbi), tile))
        else:
            rj_clase += 1
    print(f"  [{clase}] {len(aceptados_cls)}/{n_obj} en {time.time()-t0:.1f}s "
          f"(intentos={intentos}, rj_scl={rj_scl}, rj_clase={rj_clase})")
    return aceptados_cls

aceptados["vegetacion_densa"] = muestrear_aleatoria(
    "vegetacion_densa", N_POR_CLASE, lambda nd, nb: nd > 0.6)

## 7. Muestreo guiado por DAGMA: suelo_urbano

In [ ]:
RADIO_DAGMA_PX = int(RADIO_DAGMA_M / 10)
dagma_yx = [(int(np.argmin(np.abs(y_coords - row["latitud"]))),
             int(np.argmin(np.abs(x_coords - row["longitud"]))))
            for _, row in estaciones.iterrows()]

def muestrear_dagma(clase, n_obj, ndvi_max):
    aceptados_cls = []
    intentos = rj_scl = rj_clase = rj_borde = 0
    t0 = time.time()
    while len(aceptados_cls) < n_obj and intentos < n_obj * 40:
        intentos += 1
        ey, ex = dagma_yx[int(rng.integers(0, len(dagma_yx)))]
        y = ey + int(rng.integers(-RADIO_DAGMA_PX, RADIO_DAGMA_PX + 1))
        x = ex + int(rng.integers(-RADIO_DAGMA_PX, RADIO_DAGMA_PX + 1))
        if not (HALF <= y < s2.sizes["y"] - HALF and HALF <= x < s2.sizes["x"] - HALF):
            rj_borde += 1; continue
        t = int(rng.integers(0, s2.sizes["time"]))
        tile = extraer(t, y, x)
        if tile.shape != (N_BANDAS_S2, TILE_PX, TILE_PX):
            continue
        scl = scl_of(tile)
        if scl < SCL_THRESHOLD:
            rj_scl += 1; continue
        ndvi = ndvi_of(tile)
        ndbi = ndbi_of(tile)
        if ndvi < ndvi_max:
            aceptados_cls.append((Tile(
                t_idx=t, time_s2=str(s2["time"].values[t]),
                y_idx=y, x_idx=x, lat=float(y_coords[y]), lon=float(x_coords[x]),
                clase=clase, scl_pct=scl, ndvi=ndvi, ndbi=ndbi), tile))
        else:
            rj_clase += 1
    print(f"  [{clase}] {len(aceptados_cls)}/{n_obj} en {time.time()-t0:.1f}s "
          f"(intentos={intentos}, rj_scl={rj_scl}, rj_clase={rj_clase}, rj_borde={rj_borde})")
    return aceptados_cls

aceptados["suelo_urbano"] = muestrear_dagma("suelo_urbano", N_POR_CLASE, NDVI_URBANO_MAX)

In [ ]:
total = sum(len(v) for v in aceptados.values())
print(f"\nTotal: {total}/{N_POR_CLASE * len(CLASES)}")
for c, lst in aceptados.items():
    print(f"  {c}: {len(lst)}")

## 8. Contexto fisico ERA5 + MODIS por tile

In [ ]:
ERA5_BANDS = {
    "temperature_2m":           "T2m",
    "dewpoint_temperature_2m":  "Td2m",
    "u_component_of_wind_10m":  "u10",
    "v_component_of_wind_10m":  "v10",
    "boundary_layer_height":    "BLH",
    "relative_humidity_850hPa": "RH850",
    "surface_pressure":         "psurf",
    "total_precipitation":      "precip",
}
MODIS_BANDS = {
    "Optical_Depth_047": "AOD_047",
    "Optical_Depth_055": "AOD_055",
    "Column_WV":         "WV",
}

def contexto(panel, key_time, lat, lon, t_dt, band_map, prefix):
    idx = int(np.abs((times[key_time] - t_dt).total_seconds().values).argmin())
    disponibles = panel["band"].values.tolist()
    out = {}
    for src, suf in band_map.items():
        col = f"{prefix}_{suf}"
        if src not in disponibles:
            out[col] = np.nan; continue
        try:
            v = float(panel["data"].sel(band=src).isel(time=idx).sel(
                y=lat, x=lon, method="nearest").values)
            out[col] = v if np.isfinite(v) else np.nan
        except Exception:
            out[col] = np.nan
    return out

todos = [(c, t, tile) for c, lst in aceptados.items() for t, tile in lst]
t0 = time.time()
contextos = []
for c, t, _ in todos:
    t_dt = parse_time(t.time_s2)
    e = contexto(era5,  "era5",  t.lat, t.lon, t_dt, ERA5_BANDS,  "era5")
    m = contexto(modis, "modis", t.lat, t.lon, t_dt, MODIS_BANDS, "modis")
    contextos.append({**e, **m})
df_ctx = pd.DataFrame(contextos)
print(f"Contexto fisico: {time.time()-t0:.1f}s, {len(contextos)} tiles")
print("Cobertura (no-NaN/total):")
for col in df_ctx.columns:
    print(f"  {col:<14}: {df_ctx[col].notna().sum()}/{len(df_ctx)}")

## 9. Validacion + armado del meta final

In [ ]:
tiles_arr = np.stack([tile for _, _, tile in todos])
assert tiles_arr.shape == (total, N_BANDAS_S2, TILE_PX, TILE_PX)
assert tiles_arr.dtype == np.float32

meta = pd.DataFrame([{
    "clase":  c,
    "time_s2": t.time_s2,
    "lat":    t.lat,
    "lon":    t.lon,
    "ndvi":   t.ndvi,
    "ndbi":   t.ndbi,
    "scl_pct": t.scl_pct,
    "no2":    t.no2,
    "so2":    t.so2,
    "o3":     t.o3,
    "texto":  texto_de(c, t.no2, t.so2, t.o3, t.ndvi),
} for c, t, _ in todos])

meta = pd.concat([meta, df_ctx], axis=1)
print(f"tiles: {tiles_arr.shape} {tiles_arr.dtype}  ({tiles_arr.nbytes/1024**2:.1f} MB)")
print(f"meta:  {meta.shape}  cols={list(meta.columns)}")
meta.head()

## 10. Guardar (descomentar cuando N_POR_CLASE = 1000)

In [ ]:
# OUT_DIR.mkdir(parents=True, exist_ok=True)
# np.savez_compressed(OUT_DIR / "tiles_train.npz",
#                     data=tiles_arr, bands=np.array(bands_s2))
# meta.to_parquet(OUT_DIR / "tiles_meta.parquet")
# print(f"Guardado: tiles_train.npz + tiles_meta.parquet en {OUT_DIR}")